In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

DB = "`mortgage-lending-risk-and-regulatory-analytics`.default"

data = [
    ("A101", "250000", "originated", "2026-10-10 10:00:00"),
    ("A102", "180000", "denied", "2026-10-10 10:00:00"),
    ("A102", "180000", "denied", "2026-10-10 10:00:00"),
    ("A103", "300000", "originated", "2026-10-10 09:00:00"),
    ("A103", "325000", "originated", "2026-10-11 10:00:00"),
    ("A104", "200000", "denied", "2026-10-11 09:00:00"),
    ("A104", "220000", "originated", "2026-10-11 08:00:00"),
    ("A104", "220000", "originated", "2026-10-11 09:00:00"),
    ("A104", "220000", "originated", "2026-10-11 09:02:00"),
    ("A104", "220000", "originated", "2026-10-11 09:02:00"),
    ("A104", "220000", "originated", "2026-10-11 09:03:00")
]

columns = [
    "application_id",
    "loan_amount_raw",
    "action_taken",
    "source_updated_at_raw"
]

bronze_df = spark.createDataFrame(data, columns)

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{DB}.mortgage_bronze_lab")
)

display(spark.table(f"{DB}.mortgage_bronze_lab"))


In [0]:

bronze = spark.table(f"{DB}.mortgage_bronze_lab")

exact_duplicates = (
    bronze.groupBy(*bronze.columns)
          .count()
          .filter(F.col("count") > 1)
)

display(exact_duplicates)


In [0]:
unique_rows = bronze.dropDuplicates(bronze.columns)

print("Source rows:", bronze.count())
print("Rows after exact deduplication:", unique_rows.count())

In [0]:

typed = (
    unique_rows
    .withColumn(
        "loan_amount",
        F.expr("try_cast(loan_amount_raw AS DECIMAL(12,2))")
    )
    .withColumn(
        "source_updated_at",
        F.expr("try_cast(source_updated_at_raw AS TIMESTAMP)")
    )
)

typed.show()


In [0]:

invalid_rule = (
    F.col("application_id").isNull()
    | (F.trim(F.col("application_id")) == "")
    | F.col("loan_amount").isNull()
    | (F.col("loan_amount") <= 0)
    | F.col("source_updated_at").isNull()
    | F.col("action_taken").isNull()
    | (~F.col("action_taken").isin("originated", "denied"))
)

invalid_records = typed.filter(invalid_rule)
valid_records = typed.filter(~invalid_rule)

print("Invalid records:", invalid_records.count())
print("Valid records:", valid_records.count())


In [0]:

w = (
    Window.partitionBy("application_id")
          .orderBy(F.col("source_updated_at").desc())
)

ranked = valid_records.withColumn(
    "timestamp_rank",
    F.dense_rank().over(w)
)

display(
    ranked.orderBy("application_id", "timestamp_rank")
)


In [0]:
ranked = valid_records.withColumn(
    "timestamp_rank",
    F.rank().over(w)
)

display(
    ranked.orderBy("application_id", "timestamp_rank")
)

In [0]:
ranked = valid_records.withColumn(
    "timestamp_rank",
    F.row_number().over(w)
)

display(
    ranked.orderBy("application_id", "timestamp_rank")
)

In [0]:

ambiguous_ids = (
    ranked.filter(F.col("timestamp_rank") == 1)
          .groupBy("application_id")
          .count()
          .filter(F.col("count") > 1)
          .select("application_id")
)

display(ambiguous_ids)


In [0]:

# All versions of applications with unresolved conflicts
conflicts = ranked.join(
    ambiguous_ids,
    on="application_id",
    how="inner"
)

# Applications without conflicts at the latest timestamp
unambiguous = ranked.join(
    ambiguous_ids,
    on="application_id",
    how="left_anti"
)

# Current-state candidates
silver_candidates = (
    unambiguous
    .filter(F.col("timestamp_rank") == 1)
    .select(
        "application_id",
        "loan_amount",
        "action_taken",
        "source_updated_at"
    )
)

# Older records retained in Bronze, not selected as current
superseded = unambiguous.filter(
    F.col("timestamp_rank") > 1
)

# Quarantine invalid and conflicting records
quarantine = (
    invalid_records
    .withColumn(
        "quarantine_reason",
        F.lit("INVALID_FIELD")
    )
    .unionByName(
        conflicts.drop("timestamp_rank")
                 .withColumn(
                     "quarantine_reason",
                     F.lit("AMBIGUOUS_LATEST_VERSION")
                 )
    )
)

display(silver_candidates)
display(quarantine)


In [0]:
display(invalid_records)

In [0]:

metrics = {
    "source_rows": bronze.count(),
    "exact_repeat_copies": bronze.count() - unique_rows.count(),
    "invalid_rows": invalid_records.count(),
    "superseded_rows": superseded.count(),
    "conflict_rows": conflicts.count(),
    "silver_candidate_rows": silver_candidates.count()
}

accounted_for = (
    metrics["exact_repeat_copies"]
    + metrics["invalid_rows"]
    + metrics["superseded_rows"]
    + metrics["conflict_rows"]
    + metrics["silver_candidate_rows"]
)

assert metrics["source_rows"] == accounted_for, (
    "Reconciliation failed!"
)

print(metrics)
print("Reconciliation passed.")


In [0]:

(
    silver_candidates.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{DB}.mortgage_silver_candidate_lab")
)

(
    quarantine.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{DB}.mortgage_quarantine_lab")
)


In [0]:

quarantine_count = quarantine.count()

if quarantine_count > 0:
    raise RuntimeError(
        f"Silver publication blocked: "
        f"{quarantine_count} unresolved records."
    )

(
    silver_candidates.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{DB}.mortgage_silver_lab")
)
